In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import sys
from pathlib import Path

In [ ]:
# Add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline import NYCTaxiPipeline


pipe = NYCTaxiPipeline()
train_df, test_df = pipe.load_and_split()

train_ready = pipe.fit_transform(train_df)
test_ready = pipe.transform(test_df)

In [ ]:


# 1. Build the 263 Zones x 24 Hours Demand Profile Matrix
# We aggregate the training data to find the average trip count per zone for each hour of the day (0 to 23).
profile_df = (
    train_df.groupby(["zone_id", "hour"])["trip_count"]
    .mean()
    .unstack(level="hour")  # Shape: (263 zones, 24 hours)
    .fillna(0)
)

# Extract feature matrix X and zone IDs
zone_ids = profile_df.index.values
X_profiles = profile_df.values

# 2. Standardize the Demand Profiles
# This ensures that high-volume zones (like Midtown) and low-volume zones 
# are compared based on their temporal *shape* rather than absolute magnitude.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_profiles)

# 3. Evaluate K-Means Across Different Values of k Using Silhouette Score
k_range = range(3, 9)  # Testing k from 3 to 8
silhouette_scores = []
inertias = []

print("Evaluating K-Means cluster configurations...")
for k in k_range:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
  cluster_labels = kmeans.fit_predict(X_scaled)

  # Compute Silhouette Score (higher is better, measures cluster separation & cohesion)
  score = silhouette_score(X_scaled, cluster_labels)
  silhouette_scores.append(score)
  inertias.append(kmeans.inertia_)
  print(f" > k = {k} | Silhouette Score: {score:.4f}")

# 4. Select Optimal k
optimal_k = k_range[np.argmax(silhouette_scores)]
print(
    f"\n✨ Optimal k chosen by highest Silhouette Score: k = {optimal_k}"
)

# 5. Fit Final Optimal K-Means Model
optimal_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
final_labels = optimal_kmeans.fit_predict(X_scaled)

# Map cluster labels back to zones
zone_clusters = pd.DataFrame(
    {"zone_id": zone_ids, "cluster_id": final_labels}
).set_index("zone_id")